# Inspect c along the predicted D-vs-T line

这个 notebook 用现有的 `smallest_D(...)` / `find_alpha_pair(...)` 逻辑，在给定 `q`、`beta`、`c_target` 等参数后：

1. 生成不同 `T` 下预测的最小 `D` 线；
2. 显示这条线上实际返回的 `c`；
3. 可选地检查线以上若干个 `D` 的 `c`，看 `c >= c_target` 区域里的具体数值。

In [1]:
from functools import partial

import numpy as np
import matplotlib.pyplot as plt

try:
    import pandas as pd
except ImportError:
    pd = None

from qrk_analysis.feasibility.check import (
    check_feasibility_conditions_C_sup_revised,
    check_feasibility_conditions_random_sup_revised,
)
from qrk_analysis.feasibility.search import find_alpha_pair
from qrk_analysis.upper_bound import smallest_D

%matplotlib inline

## Parameters

这些默认值来自 `heatmap_generation_D_vs_T_demo.py`。如果你想看别的设置，主要改这里。

In [3]:
# Streaming system parameters
n = 100

# Noise parameters. Used by sup_c / sup_rand only.
c_min = -10
c_max = 10
s_min = 0
s_max = 10

# Algorithm / bound parameters
q = 0.8
beta = 0.005
c_target = 0.001
delta_f = 0.1
D_max = 500
D_precision = 0.1
num_grid = 50

# One of: "adversarial", "sup_c", "sup_rand"
corruption_type = "sup_c"

# Match the demo's D-vs-T predicted line: T = 100, 200, ..., 10_000.
T_intervals = 100
T_max = 10_000
T_values = np.arange(T_intervals, T_max + T_intervals, T_intervals)

# For the local grid above the predicted line, evaluate D_min + these offsets.
D_offsets_above_line = np.arange(0, 11)

In [4]:
def make_feasibility_check(corruption_type: str):
    match corruption_type:
        case "adversarial":
            return None
        case "sup_c":
            return partial(
                check_feasibility_conditions_C_sup_revised,
                num_grid_Q=2,
                C_min=c_min,
                C_max=c_max,
                num_points_C=20,
            )
        case "sup_rand":
            return partial(
                check_feasibility_conditions_random_sup_revised,
                num_grid_Q=20,
                sigma_min=s_min,
                sigma_max=s_max,
                num_points_C=50,
            )
        case _:
            raise ValueError(f"Unknown corruption_type: {corruption_type!r}")


def c_from_result(result: dict | None) -> float:
    if result is None:
        return np.nan
    return result.get("c", result.get("c_min", np.nan))


def compare_alpha_search(T: int, D: int) -> dict:
    default_pair, default_result = find_alpha_pair(
        T,
        beta,
        D,
        q,
        delta_f,
        num_grid=num_grid,
        c_target=c_target,
        feasibility_check=feasibility_check,
    )
    max_pair, max_result = find_alpha_pair(
        T,
        beta,
        D,
        q,
        delta_f,
        num_grid=num_grid,
        c_target=c_target,
        feasibility_check=feasibility_check,
        maximize_c=True,
        return_best_c_even_if_infeasible=True,
    )
    c_default = c_from_result(default_result)
    c_max = c_from_result(max_result)
    return {
        "feasible_default": default_pair is not None,
        "feasible_max": max_pair is not None,
        "c_default": c_default,
        "c_max_alpha0": c_max,
        "c_gain": c_max - c_default,
        "alpha_0_default": np.nan if default_pair is None else default_pair[0],
        "alpha_0_max": np.nan if max_pair is None else max_pair[0],
        "alpha_prime": np.nan if max_pair is None else max_pair[1],
        "failure_prob_default": np.nan if default_result is None else default_result.get("failure_prob", np.nan),
        "failure_prob_max": np.nan if max_result is None else max_result.get("failure_prob", np.nan),
    }


def display_rows(rows: list[dict]) -> None:
    if pd is not None:
        display(pd.DataFrame(rows))
        return

    columns = list(rows[0]) if rows else []
    print("\t".join(columns))
    for row in rows:
        print("\t".join(str(row[col]) for col in columns))


feasibility_check = make_feasibility_check(corruption_type)

## Predicted line: smallest D for each T

`c_smallest_D_default` 是 `smallest_D(...)` 在二分得到的最小 `D` 上返回的默认 contraction 值。`c_ceil_D_default` 和 `c_ceil_D_max_alpha0` 则是在整数 `ceil(D_min)` 上分别用默认 alpha 搜索和 `maximize_c=True` 搜索得到的值。

In [5]:
line_rows = []

for T in T_values:
    result = smallest_D(
        beta,
        int(T),
        q,
        D_max=D_max,
        D_precision=D_precision,
        delta_f=delta_f,
        c_target=c_target,
        num_grid=num_grid,
        feasibility_check=feasibility_check,
    )
    smallest_d = result.get("smallest_D")
    ceil_d = np.nan if smallest_d is None else int(np.ceil(smallest_d))
    alpha_comparison = {} if smallest_d is None else compare_alpha_search(int(T), ceil_d)
    line_rows.append(
        {
            "T": int(T),
            "D_min": np.nan if smallest_d is None else float(smallest_d),
            "ceil_D_min": ceil_d,
            "c_smallest_D_default": c_from_result(result),
            "c_ceil_D_default": alpha_comparison.get("c_default", np.nan),
            "c_ceil_D_max_alpha0": alpha_comparison.get("c_max_alpha0", np.nan),
            "c_gain": alpha_comparison.get("c_gain", np.nan),
            "c_max_minus_target": alpha_comparison.get("c_max_alpha0", np.nan) - c_target,
            "alpha_0_default": alpha_comparison.get("alpha_0_default", np.nan),
            "alpha_0_max": alpha_comparison.get("alpha_0_max", np.nan),
            "alpha_prime": alpha_comparison.get("alpha_prime", np.nan),
            "hit_ceiling": result.get("hit_ceiling", False),
        }
    )

display_rows(line_rows)

,T,D_min,ceil_D_min,c_smallest_D_default,c_ceil_D_default,c_ceil_D_max_alpha0,c_gain,c_max_minus_target,alpha_0_default,alpha_0_max,alpha_prime,hit_ceiling
0,100,2.766479,3,0.001084,0.001096,0.031313,0.030217,0.030313,0.129796,0.535408,0.001263,False
1,200,4.045654,5,0.001123,0.001131,0.046734,0.045603,0.045734,0.129796,0.567857,0.003945,False
2,300,4.715698,5,0.001129,0.001131,0.046734,0.045603,0.045734,0.129796,0.567857,0.000891,False
3,400,5.263916,6,0.001132,0.001134,0.053388,0.052254,0.052388,0.129796,0.567857,0.002256,False
4,500,5.629395,6,0.001133,0.001134,0.053388,0.052254,0.052388,0.129796,0.567857,0.000994,False
...,...,...,...,...,...,...,...,...,...,...,...,...
95,9600,10.867920,11,0.001135,0.001135,0.079347,0.078212,0.078347,0.129796,0.600306,0.000176,False
96,9700,10.928833,11,0.001135,0.001135,0.079347,0.078212,0.078347,0.129796,0.600306,0.000151,False
97,9800,10.928833,11,0.001135,0.001135,0.079347,0.078212,0.078347,0.129796,0.600306,0.000126,False
98,9900,10.928833,11,0.001135,0.001135,0.079347,0.078212,0.078347,0.129796,0.600306,0.000102,False


In [ ]:
T_plot = np.array([row["T"] for row in line_rows], dtype=float)
D_plot = np.array([row["D_min"] for row in line_rows], dtype=float)
c_default_plot = np.array([row["c_ceil_D_default"] for row in line_rows], dtype=float)
c_max_plot = np.array([row["c_ceil_D_max_alpha0"] for row in line_rows], dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(T_plot, D_plot, marker="o", linewidth=1.5, markersize=3)
axes[0].set_xlabel("T")
axes[0].set_ylabel("predicted smallest D")
axes[0].set_title("Predicted D-vs-T line")
axes[0].grid(True, alpha=0.3)

axes[1].plot(T_plot, c_default_plot, marker="o", linewidth=1.5, markersize=3, label="default alpha_0")
axes[1].plot(T_plot, c_max_plot, marker="s", linewidth=1.5, markersize=3, label="max c over alpha_0")
axes[1].axhline(c_target, color="tab:red", linestyle="--", linewidth=1.2, label="c_target")
axes[1].set_xlabel("T")
axes[1].set_ylabel("c")
axes[1].set_title("c at ceil(D_min): default vs max alpha_0")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()

## Local grid above the line

这里把每个 `T` 的 `ceil(D_min) + offset` 再丢回 `find_alpha_pair(...)` 两次：一次默认返回第一个 feasible alpha pair，一次用 `maximize_c=True` 在所有 feasible `alpha_0` 里取最大的 `c`。

In [ ]:
grid_rows = []
grid_c_default = np.full((len(line_rows), len(D_offsets_above_line)), np.nan)
grid_c_max = np.full((len(line_rows), len(D_offsets_above_line)), np.nan)

for t_idx, line_row in enumerate(line_rows):
    base_d = line_row["ceil_D_min"]
    if np.isnan(base_d):
        continue

    for offset_idx, offset in enumerate(D_offsets_above_line):
        D = int(base_d + offset)
        comparison = compare_alpha_search(line_row["T"], D)
        grid_c_default[t_idx, offset_idx] = comparison["c_default"]
        grid_c_max[t_idx, offset_idx] = comparison["c_max_alpha0"]
        grid_rows.append(
            {
                "T": line_row["T"],
                "D": D,
                "D_minus_ceil_line": int(offset),
                **comparison,
                "c_default_minus_target": comparison["c_default"] - c_target,
                "c_max_minus_target": comparison["c_max_alpha0"] - c_target,
            }
        )

display_rows(grid_rows)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5.5), sharey=True)
plot_items = [
    (grid_c_default, "default alpha_0 c", "c"),
    (grid_c_max, "max alpha_0 c", "c"),
    (grid_c_max - grid_c_default, "max - default", "c gain"),
]
tick_count = min(12, len(T_values))
tick_positions = np.linspace(0, len(T_values) - 1, tick_count, dtype=int)

for ax, (grid, title, colorbar_label) in zip(axes, plot_items):
    image = ax.imshow(
        np.ma.masked_invalid(grid),
        aspect="auto",
        origin="lower",
        interpolation="nearest",
    )
    fig.colorbar(image, ax=ax, label=colorbar_label)
    ax.set_xlabel("D offset above ceil(D_min)")
    ax.set_xticks(np.arange(len(D_offsets_above_line)))
    ax.set_xticklabels(D_offsets_above_line)
    ax.set_title(title)

axes[0].set_ylabel("T")
axes[0].set_yticks(tick_positions)
axes[0].set_yticklabels(T_values[tick_positions])
plt.tight_layout()

## Why can c be independent of T?

For fixed `D`, changing `T` changes the failure-probability budget and hence `alpha_prime`. This diagnostic checks whether that change affects the worst-case contraction. With `sup_c` and `num_grid_Q=2`, if `worst_Qq` is always `Qq_grid[0]`, then it equals `alpha_0 / (1-beta)`, which does not depend on `alpha_prime` or `T`.

In [ ]:
# Choose a D that is feasible for several T values.
diagnostic_D = 20
diagnostic_rows = []

for T in T_values:
    alpha_pair, result = find_alpha_pair(
        int(T), beta, diagnostic_D, q, delta_f,
        num_grid=num_grid,
        c_target=c_target,
        feasibility_check=feasibility_check,
        maximize_c=True,
        return_best_c_even_if_infeasible=True,
    )
    q_grid = [] if result is None else result.get("Qq_grid", [])
    c_values = [] if result is None else result.get("c_values", [])
    diagnostic_rows.append(
        {
            "T": int(T),
            "feasible": alpha_pair is not None,
            "alpha_0": np.nan if alpha_pair is None else alpha_pair[0],
            "alpha_prime": np.nan if alpha_pair is None else alpha_pair[1],
            "c_min": c_from_result(result),
            "worst_Qq": np.nan if result is None else result.get("worst_Qq", np.nan),
            "Qq_lower": np.nan if len(q_grid) == 0 else q_grid[0],
            "Qq_upper": np.nan if len(q_grid) == 0 else q_grid[-1],
            "c_at_Qq_lower": np.nan if len(c_values) == 0 else c_values[0],
            "c_at_Qq_upper": np.nan if len(c_values) == 0 else c_values[-1],
            "failure_prob": np.nan if result is None else result.get("failure_prob", np.nan),
        }
    )

display_rows(diagnostic_rows)

In [ ]:
diagnostic_T = np.array([row["T"] for row in diagnostic_rows])
alpha_prime_values = np.array([row["alpha_prime"] for row in diagnostic_rows])
c_min_values = np.array([row["c_min"] for row in diagnostic_rows])
c_lower_values = np.array([row["c_at_Qq_lower"] for row in diagnostic_rows])
c_upper_values = np.array([row["c_at_Qq_upper"] for row in diagnostic_rows])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(diagnostic_T, alpha_prime_values, marker="o")
axes[0].set_title(f"alpha_prime selected at D={diagnostic_D}")
axes[0].set_xlabel("T")
axes[0].set_ylabel("alpha_prime")
axes[0].grid(True, alpha=0.3)

axes[1].plot(diagnostic_T, c_min_values, marker="o", label="c_min")
axes[1].plot(diagnostic_T, c_lower_values, marker="s", linestyle="--", label="c at Qq_grid[0]")
axes[1].plot(diagnostic_T, c_upper_values, marker="^", linestyle="--", label="c at Qq_grid[-1]")
axes[1].set_title(f"Which Qq determines c at D={diagnostic_D}")
axes[1].set_xlabel("T")
axes[1].set_ylabel("c")
axes[1].grid(True, alpha=0.3)
axes[1].legend()
plt.tight_layout()

## Quick filters

这些 cell 用来快速看最贴近阈值的位置，或者只看不满足 `c_target` 的点。

In [ ]:
if pd is not None:
    grid_df = pd.DataFrame(grid_rows)
    display(grid_df.sort_values("c_max_minus_target").head(20))
else:
    sorted_rows = sorted(grid_rows, key=lambda row: row["c_max_minus_target"])
    display_rows(sorted_rows[:20])

In [ ]:
if pd is not None:
    below_target = grid_df[grid_df["c_max_alpha0"] < c_target]
    display(below_target)
else:
    below_target = [row for row in grid_rows if row["c_max_alpha0"] < c_target]
    display_rows(below_target)

print(f"points below c_target: {len(below_target)}")